# Custom LibriMix Training Dataset Generator

This notebook downloads the **LibriSpeech train-clean-100** source audio (~6.3 GB) and generates our own custom, high-quality **Libri2Mix** and **Libri3Mix** training datasets.

### Kaggle Disk Constraints Workflow:
Kaggle limits the `/kaggle/working` directory to **20 GB**. To generate our own data without hitting this limit:
1. We generate the datasets. We limit the 3-speaker training set to **9,000 mixtures** (~15 GB) to fit safely within the disk limit.
2. Run this notebook using **Save Version -> Save & Run All (Commit)**.
3. Once finished, B and C can attach the **Output of this notebook** to their training notebooks as a read-only input. This mounts the dataset at `/kaggle/input/` and bypasses the 20 GB limit during training!

In [ ]:
!pip -q install pandas numpy soundfile 2>/dev/null
print('Setup complete.')

### 1. Locate Source Scripts

In [ ]:
import sys, os, glob
hits = glob.glob('/kaggle/**/make_libri3mix_test.py', recursive=True)
assert hits, 'make_libri3mix_test.py not found under /kaggle -- is the utility dataset attached?'
SRC_DIR = os.path.dirname(hits[0])
print('Using source scripts from:', SRC_DIR)

### 2. Download Training Source Audio & Clones Metadata

In [ ]:
%cd /kaggle/working
# Download train-clean-100 (6.3 GB)
print('Downloading LibriSpeech train-clean-100...')
!wget -c -q https://www.openslr.org/resources/12/train-clean-100.tar.gz
print('Extracting source files...')
!tar -xzf train-clean-100.tar.gz

# Clone metadata
!git clone -q https://github.com/JorisCos/LibriMix
print('Extraction and metadata clone complete.')

### 3. Choose Dataset to Generate
Toggle which dataset to generate. Do **not** generate both in the same run to avoid exceeding the 20 GB disk limit.

In [ ]:
# Set variables
GENERATE_3_SPEAKER = True  # Set to True for Libri3Mix, False for Libri2Mix
LIMIT = 9000              # Number of mixtures. 9,000 for Libri3Mix fits ~15 GB output limit.

### 4. Execute Generator

In [ ]:
if GENERATE_3_SPEAKER:
    print('Generating Libri3Mix training dataset...')
    !python {SRC_DIR}/make_libri3mix_test.py \
        --metadata LibriMix/metadata/Libri3Mix/libri3mix_train-100.csv \
        --librispeech-root LibriSpeech \
        --outdir libri3mix_train \
        --sr 8000 \
        --limit {LIMIT}
else:
    print('Generating Libri2Mix training dataset...')
    !python {SRC_DIR}/make_libri3mix_test.py \
        --metadata LibriMix/metadata/Libri2Mix/libri2mix_train-100.csv \
        --librispeech-root LibriSpeech \
        --outdir libri2mix_train \
        --sr 8000

### 5. Clean up downloaded source files to free space

In [ ]:
# Remove the downloaded tar.gz and source LibriSpeech to save output disk space
!rm -f train-clean-100.tar.gz
!rm -rf LibriSpeech
!rm -rf LibriMix
print('Cleanup complete. Only output dataset remains in working directory.')